# 03 — Paper Trading Dashboard (MAIN)

Live cross-venue arb scanner + paper executor + portfolio dashboard.

Open this notebook and run all cells top-to-bottom. The scanner will:
1. Pull markets from both platforms (refreshed periodically)
2. Run the matcher
3. Poll prices for matched pairs every N seconds
4. Detect arbs
5. Auto-execute in paper mode (no real money)
6. Display a live, color-coded table

**Stop with Jupyter's interrupt-kernel (■) button.**


## Setup


In [7]:
import asyncio, sys, time, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
import yaml
from IPython.display import clear_output, display
from clients.kalshi_client import KalshiClient
from clients.polymarket_client import PolymarketClient
from core.event_matcher import EventMatcher, flatten_market_pairs
from core.arb_detector import detect_all
from core.portfolio import Portfolio
from core.risk_manager import RiskManager
from core.execution_engine import ExecutionEngine
from data.models import Platform

CFG = yaml.safe_load(open("../config.yaml"))

KALSHI_SERIES = [
    "KXPRES", "KXSEN", "KXHOUSE", "KXIMPEACH",
    "KXBTCD", "KXETHD", "KXBTC", "KXETH", "KXSOL",
    "KXFEDDECISION", "KXCPI", "KXJOBS",
    "KXSPX", "KXNASDAQ",
    "KXWORLDCUP", "KXNBA", "KXWAR",
]
print("Loaded config. Mode:", CFG["mode"])


Loaded config. Mode: paper


In [8]:
k = KalshiClient(environment=CFG["kalshi"]["environment"])
p = PolymarketClient()
print(f"Kalshi authed={k._authed}  Polymarket authed={p._authed}")


INFO     KalshiClient init  env=production  authed=True

INFO     PolymarketClient init  authed=False

Kalshi authed=True  Polymarket authed=False


## Pull markets + build match cache


In [9]:
matcher = EventMatcher(
    min_event_score=0.45,
    strike_tolerance_pct=0.02,
    max_end_date_delta_days=14,
)
k_events, p_events = await asyncio.gather(
    k.get_events_with_markets(KALSHI_SERIES, limit_per_series=30),
    p.get_events_with_markets(limit=300),
)
event_pairs = matcher.match(k_events, p_events)
pairs = flatten_market_pairs(event_pairs)
print(f"K events={len(k_events)}  P events={len(p_events)}")
print(f"Event-level matches: {len(event_pairs)}")
print(f"Market-level pairs:  {len(pairs)}")

# Show top event matches
for ep in event_pairs[:5]:
    n = len(ep.market_pairs)
    print(f"  [{ep.method.value:>6}] conf={ep.confidence:.2f}  {n} mkts  "
          f"K:'{ep.kalshi_event.title[:40]}' ↔ P:'{ep.polymarket_event.title[:40]}'")

# Filter to high-confidence pairs for scanning
PAIRS_TO_SCAN = [pp for pp in pairs if pp.confidence >= 0.65][:100]
print(f"\nScanning {len(PAIRS_TO_SCAN)} pairs each cycle")


INFO     event_pairs=8 (dropped 5 with 0 child pairs)  total_market_pairs=67

K events=36  P events=300
Event-level matches: 8
Market-level pairs:  67
  [ fuzzy] conf=0.84  11 mkts  K:'Bitcoin price on May 22, 2026 at 5pm EDT' ↔ P:'Bitcoin above ___ on May 22?'
  [ fuzzy] conf=0.84  11 mkts  K:'Ethereum price on May 22, 2026 at 5pm ED' ↔ P:'Ethereum above ___ on May 22?'
  [ fuzzy] conf=0.76  7 mkts  K:'Ethereum price on May 18, 2026 at 4pm ED' ↔ P:'What price will Ethereum hit May 18-24?'
  [ fuzzy] conf=0.77  14 mkts  K:'Bitcoin price range on May 18, 2026 at 4' ↔ P:'What price will Bitcoin hit May 18-24?'
  [ fuzzy] conf=0.75  14 mkts  K:'Ethereum price range on May 18, 2026 at ' ↔ P:'What price will Ethereum hit on May 18?'

Scanning 43 pairs each cycle


## Portfolio, risk, execution engine


In [10]:
portfolio = Portfolio(
    db_path="../" + CFG["paper"]["db_path"],
    starting_capital_usd=CFG["paper"]["starting_capital_usd"],
)
risk = RiskManager(
    max_position_per_market_usd=CFG["risk"]["max_position_per_market_usd"],
    max_total_exposure_usd=CFG["risk"]["max_total_exposure_usd"],
    max_daily_loss_usd=CFG["risk"]["max_daily_loss_usd"],
    min_net_edge_cents=CFG["arbitrage"]["min_net_edge_cents"],
    min_match_confidence=CFG["risk"]["min_match_confidence"],
    min_liquidity_usd=CFG["arbitrage"]["min_liquidity_usd"],
)
engine = ExecutionEngine(k, p, portfolio, risk, mode="paper")
print("Engine ready in PAPER mode.")
print(f"Starting capital: ${portfolio.starting_capital:,.2f}")
print(f"Current cash:     ${portfolio.cash_usd():,.2f}")


Engine ready in PAPER mode.
Starting capital: $5,000.00
Current cash:     $5,000.00


## Live scanner + executor

Runs continuously. Pulls prices every `POLL_INTERVAL` seconds, detects arbs,
auto-executes in paper mode. Updates a colored table in place.


In [11]:
async def pull_prices(pairs):
    """Pull current top-of-book for both legs of each pair.

    CRITICAL: pass the explicit yes_token_id when querying Polymarket so we
    hit the right book directly. Without it, the get_price fallback path
    can return the same (wrong) market data for every pair.
    """
    tasks = []
    for pp in pairs:
        tasks.append(k.get_price(pp.kalshi_market.market_id))
        tasks.append(p.get_price(pp.polymarket_market.market_id,
                                  token_id=pp.polymarket_market.yes_token_id))
    results = await asyncio.gather(*tasks, return_exceptions=True)
    prices = {}
    for i, pp in enumerate(pairs):
        kr = results[2*i]; pr = results[2*i+1]
        if not isinstance(kr, Exception):
            prices[(Platform.KALSHI, pp.kalshi_market.market_id)] = kr
        if not isinstance(pr, Exception):
            prices[(Platform.POLYMARKET, pp.polymarket_market.market_id)] = pr
    return prices

def style_row(row):
    net = row.get("net_c")
    if net is None:
        return [""] * len(row)
    if net >= CFG["arbitrage"]["min_net_edge_cents"]:
        return ["background-color: #d4edda"] * len(row)   # green
    if net >= 0:
        return ["background-color: #fff3cd"] * len(row)   # yellow
    return [""] * len(row)


In [12]:
POLL_INTERVAL = CFG["scanning"]["poll_interval_seconds"]
RUN_CYCLES = 15   # ~30 min at 30s interval

for cycle in range(RUN_CYCLES):
    t0 = time.time()
    prices = await pull_prices(PAIRS_TO_SCAN)
    ops = detect_all(
        PAIRS_TO_SCAN, prices,
        min_net_edge_cents=-5.0,  # show even negative for the live table
        min_liquidity_usd=CFG["arbitrage"]["min_liquidity_usd"],
        max_capital_usd=CFG["risk"]["max_position_per_market_usd"],
        max_slippage_pct=CFG["arbitrage"]["max_slippage_pct"],
    )
    # Auto-execute positive-net opportunities
    for op in ops:
        if op.net_edge_cents >= CFG["arbitrage"]["min_net_edge_cents"]:
            await engine.execute(op)
    # Build table
    rows = []
    for pp in PAIRS_TO_SCAN:
        kp = prices.get((Platform.KALSHI, pp.kalshi_market.market_id))
        pr = prices.get((Platform.POLYMARKET, pp.polymarket_market.market_id))
        if not (kp and pr): continue
        # Find matching arb op for this pair
        op = next((o for o in ops if o.pair is pp), None)
        rows.append({
            "kalshi_id": pp.kalshi_market.market_id[:30],
            "kalshi_title": pp.kalshi_market.title[:45],
            "K_yes_ask": round(kp.yes_ask or 0, 4),
            "P_yes_ask": round(pr.yes_ask or 0, 4),
            "K_no_ask": round(kp.no_ask or 0, 4),
            "P_no_ask": round(pr.no_ask or 0, 4),
            "gross_c": round(op.gross_edge_cents, 2) if op else None,
            "net_c":   round(op.net_edge_cents, 2) if op else None,
            "size":    round(op.max_size_contracts, 0) if op else None,
            "signal":  "ARB" if (op and op.net_edge_cents >= CFG["arbitrage"]["min_net_edge_cents"])
                       else ("WATCH" if op and op.net_edge_cents >= 0 else ""),
        })
    df = pd.DataFrame(rows).sort_values("net_c", ascending=False, na_position="last")

    # Portfolio snapshot
    snap = portfolio.snapshot()

    clear_output(wait=True)
    print(f"══ Cycle {cycle+1}/{RUN_CYCLES}  ({time.strftime('%H:%M:%S')}) ══")
    print(f"  Pairs scanned: {len(rows)}  Arb signals: {sum(1 for r in rows if r['signal']=='ARB')}")
    print(f"  Cash: ${snap.cash_usd:,.2f}  Locked: ${snap.locked_in_positions_usd:,.2f}  "
          f"Open arbs: {snap.open_arb_pairs}  Realized: ${snap.realized_pnl_usd:+,.2f}")
    print(f"  Kill switch: {'YES' if risk.kill_switch else 'no'}")
    display(df.head(20).style.apply(style_row, axis=1))

    # Sleep
    elapsed = time.time() - t0
    await asyncio.sleep(max(0, POLL_INTERVAL - elapsed))

print("Scanner stopped.")


══ Cycle 15/15  (14:54:12) ══
  Pairs scanned: 41  Arb signals: 1
  Cash: $5,000.00  Locked: $0.00  Open arbs: 0  Realized: $+0.00
  Kill switch: no


,kalshi_id,kalshi_title,K_yes_ask,P_yes_ask,K_no_ask,P_no_ask,gross_c,net_c,size,signal
30,KXBTC-26MAY1816-B72650,"Bitcoin price range on May 18, 2026?",0.010000,0.990000,0.000000,0.600000,39.000000,36.030000,750.000000,ARB
18,KXBTC-26MAY1816-T68700,"Bitcoin price range on May 18, 2026?",0.010000,0.999000,0.000000,0.970000,2.000000,-0.970000,114.000000,
10,KXBTCD-26MAY2217-T68999.99,"Bitcoin price on May 22, 2026?",0.990000,0.999000,0.040000,0.026000,-1.600000,-1.630000,100.000000,
22,KXETH-26MAY1817-B2050,"Ethereum price at May 18, 2026 at 5pm EDT?",0.030000,0.990000,0.990000,0.970000,0.000000,-2.910000,198.000000,
4,KXETH-26MAY1817-B1970,"Ethereum price at May 18, 2026 at 5pm EDT?",0.010000,0.999000,0.000000,0.990000,0.000000,-2.970000,201.000000,
5,KXETHD-26MAY2217-T2569.99,"Ethereum price at May 22, 2026 at 5pm EDT?",0.010000,0.999000,0.000000,0.997000,-0.700000,-3.670000,100.000000,
20,KXETH-26MAY1817-B2250,"Ethereum price at May 18, 2026 at 5pm EDT?",0.010000,0.999000,0.000000,0.997000,-0.700000,-3.670000,201.000000,
9,KXBTCD-26MAY2217-T70999.99,"Bitcoin price on May 22, 2026?",0.970000,0.999000,0.050000,0.066000,-3.600000,-3.690000,100.000000,
21,KXETHD-26MAY2217-T2449.99,"Ethereum price at May 22, 2026 at 5pm EDT?",0.020000,0.999000,0.000000,0.997000,-1.700000,-4.640000,100.000000,
17,KXETHD-26MAY2217-T2649.99,"Ethereum price at May 22, 2026 at 5pm EDT?",0.020000,0.999000,0.000000,0.998000,-1.800000,-4.740000,200.000000,


Scanner stopped.


## Portfolio dashboard


In [ ]:
snap = portfolio.snapshot()
print(f"══ PORTFOLIO ══")
print(f"  Starting:       ${snap.starting_capital_usd:,.2f}")
print(f"  Cash:           ${snap.cash_usd:,.2f}")
print(f"  Locked:         ${snap.locked_in_positions_usd:,.2f}")
print(f"  Realized PnL:   ${snap.realized_pnl_usd:+,.2f}")
print(f"  Unrealized:     ${snap.unrealized_pnl_usd:+,.2f}")
print(f"  Equity:         ${snap.equity_usd:,.2f}")
print(f"  Total return:   {snap.total_return_pct:+.2f}%")
print(f"  Open arbs:      {snap.open_arb_pairs}")
print(f"  Settled arbs:   {snap.settled_arb_pairs}  ({snap.wins}W/{snap.losses}L)")


In [ ]:
# Open arbs in detail
df_open = portfolio.open_arb_pairs_df()
df_open


In [ ]:
# Recent events
portfolio.all_events_df().head(20)


## Analytics


In [ ]:
import plotly.graph_objects as go
events = portfolio.all_events_df()
import json as _json
opens = events[events["event_type"] == "open_arb"].copy()
if len(opens):
    opens["edge_usd"] = opens["details"].apply(lambda s: _json.loads(s).get("expected_edge_usd", 0))
    fig = go.Figure(go.Histogram(x=opens["edge_usd"], nbinsx=30))
    fig.update_layout(title="Distribution of expected edges captured (USD)",
                       xaxis_title="Expected net edge ($)", yaxis_title="Trades",
                       height=400)
    fig.show()
else:
    print("No trades yet.")


## Cleanup


In [ ]:
await k.close(); await p.close()
print("Closed. Run the scanner cell again any time to resume.")
